# XGBoost. Прогноз остатка

Обучаем отдельную модель для каждого из восьми горизонтов на всех маршрутах вместе. Цель равна `(y - profile28) / 100000` с добавлением профиля при прогнозировании. Отрицательные прогнозы ограничиваются нулём.

Train использует пять стартов с 09:30 до 11:30. Val, подбор, test и конкурсный прогноз используют 10:30.

## Подбор и проверка

Optuna сравнивает параметры на трёх недельных окнах с 30 августа по 19 сентября 2025 года. Для каждого горизонта выбирается минимум средней WAPE + |Relative Bias| трёх окон. После выбора модель обучается на истории до test и оценивается на 18-31 октября.

Заполнение пропусков, масштабирование и корреляционные лаги оцениваются по train текущего окна. Начальные строки без доступного `profile28` исключаются. Параметры из первой ячейки задают число попыток и устройство.

Перед сабмитом модели переобучаются на всей доступной истории. Независимая оценка test получена до этого переобучения. Период test уже использовался в проекте.

## Результаты

В `artifacts/<модель>_<direct или residual>_stack/` сохраняются признаки, модели, прогнозы, отчёты и параметры `model_params/best_hN.json`.

Подробный протокол приведён в [описании экспериментов](../../docs/model_experiments.md).


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import gc
import os
import importlib.util
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
ALGORITHM = 'xgb'
RESIDUAL = True
DEVICE = 'cuda' if ALGORITHM in ('xgb', 'catboost') else 'cpu'
N_TRIALS = 20
N_JOBS = min(4, os.cpu_count() or 1)
SEED = 42
MAX_TREES = 3000
EARLY_STOPPING_ROUNDS = 100
PREDICTION_BATCH = 4096
RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
required_modules = ['sklearn', 'optuna', 'joblib', 'psutil']
required_modules += [{'xgb': 'xgboost', 'catboost': 'catboost', 'lightgbm': 'lightgbm', 'prophet': 'prophet'}.get(ALGORITHM, 'sklearn')]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Установите недостающие пакеты до расчёта признаков: {sorted(set(missing))}')
if N_TRIALS < 1 or PREDICTION_BATCH < 1:
    raise ValueError('Число trials и размер блока должны быть положительными')
DATA = Path('data')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
TRAIN_PATH, TEST_PATH = DATA / TRAIN_FILE, DATA / TEST_FILE
if not (TRAIN_PATH.is_file() and TEST_PATH.is_file()) and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    test_matches = list(Path('/kaggle/input').rglob(TEST_FILE))
    if len(train_matches) != 1 or len(test_matches) != 1:
        raise ValueError('Укажите TRAIN_PATH и TEST_PATH вручную')
    TRAIN_PATH, TEST_PATH = train_matches[0], test_matches[0]
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MODEL_NAME = 'xgb_residual'
RUN_DIR = BASE / 'artifacts' / f'{MODEL_NAME}_stack'
if RUN_DIR.exists():
    raise FileExistsError(f'Папка {RUN_DIR} уже существует. Перенесите предыдущий результат или измените MODEL_NAME.')
OUT = RUN_DIR / 'features'
MODEL_DIR = RUN_DIR / 'models'
IMPORTANCE_DIR = RUN_DIR / 'importance'
POSTPROCESS_DIR = RUN_DIR / 'postprocessing'
DOCS_DIR = RUN_DIR / 'docs'
PARAMS_DIR = RUN_DIR / 'model_params'
for directory in (OUT, MODEL_DIR, IMPORTANCE_DIR, POSTPROCESS_DIR, DOCS_DIR, PARAMS_DIR):
    directory.mkdir(parents=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = 'five_starts_windows_v4_stage_safe_hpo'


## Данные и временное разбиение


In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
selection_end = pd.Timestamp('2025-09-20')
if len(dates) < 14 or dates[-14].normalize() < selection_end or times[-1] < selection_end:
    raise ValueError('Недостаточно истории для отдельного test после подбора')
holdout_start = dates[-14]
selection_dates = dates[dates + 8 * STEP < selection_end]
if len(selection_dates) <= 28:
    raise ValueError('До 20 сентября недостаточно истории для train и трёх недель подбора')
folds = []
for start in range(len(selection_dates) - 21, len(selection_dates), 7):
    folds.append(dict(train=selection_dates[:start-7], val=selection_dates[start-7:start],
                      tuning=selection_dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))

## Общие признаки пяти стартов


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

## Корреляционные лаги, обучаемые внутри train


In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

## Матрицы восьми горизонтов


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h * STEP)]
    future = requests.merge(features, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError('Ошибка сопоставления тестовых строк')
    future.to_parquet(OUT / f'submission_h{h}.parquet', index=False)
    print(f'h={h}: история {len(origins)*len(routes):,}, submission {len(future):,}, модельных признаков {len(frame.columns) - 5}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(DOCS_DIR / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origins_minutes': TRAIN_MINUTES, 'evaluation_origin': '10:30', 'correlation_storage': 'NaN placeholders; fit on stage train before model use', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'selection_end_exclusive': str(selection_end), 'all_folds': [{key: [str(period.min()), str(period.max())] for key, period in fold.items()} for fold in folds], 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))

## Метрика


In [ ]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

## Настройки моделей


In [ ]:
import importlib.metadata
import platform
import zipfile
import joblib
import optuna
import psutil
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor


def offset(frame):
    return frame.profile28.to_numpy(dtype=float) if RESIDUAL else np.zeros(len(frame))


def model_target(frame):
    return (frame.y.to_numpy(dtype=float) - offset(frame)) / TARGET_SCALE


def route_bit_columns():
    return [f'route_bit_{i}' for i in range(max(1, int(len(routes)).bit_length()))]


def transformed_frame(frame, columns):
    result = frame[columns].copy()
    if ALGORITHM in ('xgb', 'lightgbm'):
        result['route_id'] = pd.Categorical(result.route_id, categories=routes)
    elif ALGORITHM == 'catboost':
        result['route_id'] = result.route_id.astype(str)
    else:
        codes = pd.Index(routes).get_indexer(result.route_id).astype(np.int64) + 1
        result = result.drop(columns='route_id')
        for i, name in enumerate(route_bit_columns()):
            result[name] = ((codes >> i) & 1).astype(np.float32)
    return result


def preprocessing(columns, scale):
    numeric = [c for c in columns if c != 'route_id']
    steps = [('impute', SimpleImputer(strategy='median', keep_empty_features=True))]
    if scale:
        steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('numeric', Pipeline(steps), numeric),
        ('route', 'passthrough', route_bit_columns())
    ], sparse_threshold=0.0)


def suggest_params(trial):
    if ALGORITHM == 'xgb':
        return dict(max_depth=trial.suggest_int('max_depth', 3, 9),
                    learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
                    min_child_weight=trial.suggest_float('min_child_weight', 1, 100, log=True),
                    subsample=trial.suggest_float('subsample', 0.65, 1),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1),
                    reg_alpha=trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
                    reg_lambda=trial.suggest_float('reg_lambda', 0.1, 100, log=True))
    if ALGORITHM == 'lightgbm':
        return dict(num_leaves=trial.suggest_int('num_leaves', 15, 127, log=True),
                    learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
                    min_child_samples=trial.suggest_int('min_child_samples', 20, 200, log=True),
                    subsample=trial.suggest_float('subsample', 0.65, 1),
                    colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1),
                    reg_alpha=trial.suggest_float('reg_alpha', 1e-5, 10, log=True),
                    reg_lambda=trial.suggest_float('reg_lambda', 0.1, 100, log=True))
    if ALGORITHM == 'catboost':
        return dict(depth=trial.suggest_int('depth', 4, 8),
                    learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
                    l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 100, log=True),
                    random_strength=trial.suggest_float('random_strength', 0, 3),
                    bagging_temperature=trial.suggest_float('bagging_temperature', 0, 2))
    if ALGORITHM == 'random_forest':
        return dict(n_estimators=trial.suggest_categorical('n_estimators', [150, 300, 500]),
                    max_depth=trial.suggest_categorical('max_depth', [10, 16, 24]),
                    min_samples_leaf=trial.suggest_int('min_samples_leaf', 2, 64, log=True),
                    max_features=trial.suggest_float('max_features', 0.25, 0.9),
                    max_samples=trial.suggest_float('max_samples', 0.5, 1))
    if ALGORITHM == 'ridge':
        return dict(alpha=trial.suggest_float('alpha', 1e-3, 1e6, log=True))
    return dict(changepoint_prior_scale=trial.suggest_float('changepoint_prior_scale', 0.001, 0.2, log=True),
                seasonality_prior_scale=trial.suggest_float('seasonality_prior_scale', 0.01, 10, log=True),
                regressor_prior_scale=trial.suggest_float('regressor_prior_scale', 0.001, 10, log=True),
                n_changepoints=trial.suggest_categorical('n_changepoints', [5, 10, 20]))


def prophet_matrix(frame, columns, prep, names):
    encoded = prep.transform(transformed_frame(frame, columns))
    if hasattr(encoded, 'toarray'):
        encoded = encoded.toarray()
    result = pd.DataFrame(encoded, columns=names)
    result.insert(0, 'ds', frame.timestamp.to_numpy())
    return result


def fit_model(frame, columns, params, early=None, trees=None):
    params = dict(params)
    if len(frame) == 0 or not np.isfinite(frame.y.to_numpy()).all():
        raise ValueError('Обучающая выборка пуста или содержит некорректные метки')
    if early is not None and (len(early) == 0 or frame.timestamp.max() >= early.cutoff.min().normalize()):
        raise ValueError('Нарушена временная граница train/val')
    y = model_target(frame)
    if not np.isfinite(y).all():
        raise ValueError('Некорректная цель после преобразования')
    if ALGORITHM == 'xgb':
        from xgboost import XGBRegressor
        options = dict(n_estimators=trees or MAX_TREES, device=DEVICE, tree_method='hist',
                       enable_categorical=True, max_cat_to_onehot=1, objective='reg:squarederror', eval_metric='mae',
                       random_state=SEED, n_jobs=N_JOBS, **params)
        kwargs = {}
        if early is not None:
            options['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
            kwargs['eval_set'] = [(transformed_frame(early, columns), model_target(early))]
        model = XGBRegressor(**options)
        model.fit(transformed_frame(frame, columns), y, verbose=False, **kwargs)
        actual_device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
        if DEVICE == 'cuda' and not actual_device.startswith('cuda'):
            raise RuntimeError('Запрошен GPU, но XGBoost работает на CPU. Проверьте CUDA или явно выберите DEVICE=cpu.')
        count = model.best_iteration + 1 if early is not None else options['n_estimators']
        return dict(model=model, columns=columns, trees=int(count))
    if ALGORITHM == 'lightgbm':
        from lightgbm import LGBMRegressor, early_stopping, log_evaluation
        model = LGBMRegressor(n_estimators=trees or MAX_TREES, objective='regression', metric='l1',
                              device_type=DEVICE, subsample_freq=1, importance_type='gain',
                              max_cat_to_onehot=1, verbosity=-1, random_state=SEED, n_jobs=N_JOBS, **params)
        kwargs = dict(categorical_feature=['route_id'], callbacks=[log_evaluation(0)])
        if early is not None:
            kwargs['eval_set'] = [(transformed_frame(early, columns), model_target(early))]
            kwargs['callbacks'].append(early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False))
        model.fit(transformed_frame(frame, columns), y, **kwargs)
        count = model.best_iteration_ if early is not None else model.n_estimators_
        return dict(model=model, columns=columns, trees=int(count))
    if ALGORITHM == 'catboost':
        from catboost import CatBoostRegressor
        model = CatBoostRegressor(iterations=trees or MAX_TREES, loss_function='RMSE', eval_metric='MAE',
                                  task_type='GPU' if DEVICE == 'cuda' else 'CPU',
                                  thread_count=N_JOBS, random_seed=SEED, bootstrap_type='Bayesian',
                                  allow_writing_files=False, one_hot_max_size=1, verbose=False, **params)
        kwargs = {}
        if early is not None:
            kwargs.update(eval_set=(transformed_frame(early, columns), model_target(early)),
                          early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
        model.fit(transformed_frame(frame, columns), y, cat_features=['route_id'], **kwargs)
        return dict(model=model, columns=columns, trees=int(model.tree_count_))
    if ALGORITHM in ('ridge', 'random_forest'):
        estimator = Ridge(solver='lsqr', tol=1e-5, **params) if ALGORITHM == 'ridge' else RandomForestRegressor(n_jobs=N_JOBS, random_state=SEED, **params)
        model = Pipeline([('preprocess', preprocessing(columns, ALGORITHM == 'ridge')), ('model', estimator)])
        model.fit(transformed_frame(frame, columns), y)
        return dict(model=model, columns=columns, trees=None)
    from prophet import Prophet
    width = len(columns) - 1 + len(route_bit_columns())
    estimated_gb = len(frame) * width * 8 * 6 / 1024 ** 3
    available_gb = psutil.virtual_memory().available / 1024 ** 3
    print(f'Prophet: оценка рабочей памяти {estimated_gb:.1f} ГБ; доступно {available_gb:.1f} ГБ')
    if estimated_gb > available_gb * 0.8:
        raise MemoryError('Для полного Prophet с категориальными маршрутами нужна машина с большей RAM. Строки и признаки автоматически не сокращаются.')
    prep = preprocessing(columns, True)
    prep.fit(transformed_frame(frame, columns))
    names = [f'x{i}' for i in range(len(prep.get_feature_names_out()))]
    data = prophet_matrix(frame, columns, prep, names)
    active = [name for name in names if data[name].nunique() > 1]
    prior = params.pop('regressor_prior_scale')
    model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True,
                    uncertainty_samples=0, **params)
    for name in active:
        model.add_regressor(name, prior_scale=prior, standardize=False)
    data = data[['ds', *active]].copy()
    data['y'] = y
    model.fit(data, seed=SEED)
    return dict(model=model, columns=columns, trees=None, prep=prep, names=names, active=active)


def predict(bundle, frame):
    chunks = []
    for start in range(0, len(frame), PREDICTION_BATCH):
        part = frame.iloc[start:start + PREDICTION_BATCH]
        if ALGORITHM == 'prophet':
            data = prophet_matrix(part, bundle['columns'], bundle['prep'], bundle['names'])
            order = np.argsort(data.ds.to_numpy(), kind='stable')
            forecast = bundle['model'].predict(data.iloc[order][['ds', *bundle['active']]])['yhat'].to_numpy()
            raw = np.empty(len(part), dtype=np.float64)
            raw[order] = forecast
        else:
            raw = bundle['model'].predict(transformed_frame(part, bundle['columns']))
        raw = np.asarray(raw, dtype=np.float64)
        restored = offset(part) + TARGET_SCALE * raw
        if raw.shape != (len(part),) or not np.isfinite(restored).all():
            raise ValueError('Модель вернула некорректные прогнозы')
        chunks.append(np.maximum(0, restored))
    return np.concatenate(chunks)


def save_bundle(bundle, folder):
    folder.mkdir(parents=True, exist_ok=True)
    metadata = dict(algorithm=ALGORITHM, residual=RESIDUAL, target_scale=TARGET_SCALE,
                    columns=bundle['columns'], trees=bundle['trees'], route_categories=routes.tolist(),
                    route_encoding='native_categorical' if ALGORITHM in ('xgb', 'catboost', 'lightgbm') else 'binary_index_plus_one',
                    route_bit_columns=route_bit_columns() if ALGORITHM not in ('xgb', 'catboost', 'lightgbm') else [])
    if ALGORITHM == 'xgb':
        bundle['model'].save_model(folder / 'model.ubj')
    elif ALGORITHM == 'lightgbm':
        bundle['model'].booster_.save_model(str(folder / 'model.txt'))
    elif ALGORITHM == 'catboost':
        bundle['model'].save_model(str(folder / 'model.cbm'))
    elif ALGORITHM == 'prophet':
        from prophet.serialize import model_to_json
        (folder / 'model.json').write_text(model_to_json(bundle['model']))
        joblib.dump({k: bundle[k] for k in ('prep', 'names', 'active')}, folder / 'preprocess.joblib', compress=3)
    else:
        joblib.dump(bundle['model'], folder / 'model.joblib', compress=3)
    (folder / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))


def add_report(rows, phase, h, trial=None, fold=None):
    records = [dict(phase=phase, horizon=h, trial=trial, fold=fold, **row) for row in rows]
    pd.DataFrame(records).to_csv(DOCS_DIR / 'metrics.csv', mode='a', header=not (DOCS_DIR / 'metrics.csv').exists(), index=False)
    display(pd.DataFrame(records))


def prediction_part(frame, prediction, h, period):
    part = frame[['cutoff', 'timestamp', 'route_id', 'y', 'profile28']].copy()
    part['prediction'] = prediction
    part[ALGORITHM] = prediction
    part['horizon'] = h
    part['period'] = period
    return part


def export_results():
    (DOCS_DIR / 'evaluation_note.txt').write_text(manifest['test_note'])
    with zipfile.ZipFile(RUN_DIR / 'reports.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in (DOCS_DIR, PARAMS_DIR, IMPORTANCE_DIR):
            for path in folder.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(RUN_DIR))
    with zipfile.ZipFile(RUN_DIR / 'results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in RUN_DIR.rglob('*'):
            if path.is_file() and path.suffix != '.zip' and OUT not in path.parents:
                archive.write(path, path.relative_to(RUN_DIR))


versions = {}
for name in ('numpy', 'pandas', 'pyarrow', 'scikit-learn', 'xgboost', 'catboost', 'lightgbm', 'prophet', 'optuna', 'joblib'):
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        pass
manifest = dict(algorithm=ALGORITHM, residual=RESIDUAL, seed=SEED, device=DEVICE, n_trials=N_TRIALS,
                python=platform.python_version(), platform=platform.platform(), versions=versions,
                feature_pipeline=FEATURE_PIPELINE_VERSION, starts=TRAIN_MINUTES,
                holdout_start=str(holdout_start), selection_end_exclusive=str(selection_end), search_objective='mean of three tuning WAPE + absolute relative bias scores',
                test_note='Независимая оценка test получена до переобучения на всей истории. Период test ранее использовался в проекте; это не новый слепой holdout.')
(DOCS_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
lag_tables = [fit_lags(f['val'][0].normalize()) for f in folds]
lag_tables[-1].to_csv(DOCS_DIR / 'status6_lags_validation.csv', index=False)
for i, table in enumerate(lag_tables):
    table.to_csv(DOCS_DIR / f'status6_lags_fold_{i+1}.csv', index=False)


def select_horizon(h):
    history = pd.read_parquet(OUT / f'history_h{h}.parquet', filters=[('timestamp', '<', selection_end)])
    history = history.loc[np.isfinite(history.profile28)].copy()
    columns = model_columns(history)
    def objective(trial):
        params = suggest_params(trial)
        metrics, tree_counts, tuning_parts = [], [], []
        for i, (fold, lag_table) in enumerate(zip(folds, lag_tables)):
            train = history.loc[history.cutoff.dt.normalize().isin(fold['train'].normalize()) & (history.timestamp < fold['val'][0].normalize())].copy()
            early = history.loc[history.cutoff.isin(fold['val'])].copy()
            tuning = history.loc[history.cutoff.isin(fold['tuning'])].copy()
            if len(train) == 0 or len(early) == 0 or len(tuning) == 0:
                raise ValueError('Пустая временная выборка')
            if train.timestamp.max() >= early.cutoff.min().normalize() or early.timestamp.max() >= tuning.cutoff.min().normalize() or tuning.timestamp.max() >= selection_end:
                raise ValueError('Пересекаются временные периоды train/val/tuning/test')
            train = add_correlation_features(train, h, lag_table)
            early = add_correlation_features(early, h, lag_table)
            tuning = add_correlation_features(tuning, h, lag_table)
            bundle = fit_model(train, columns, params, early if ALGORITHM in ('xgb', 'catboost', 'lightgbm') else None)
            tuning_prediction = predict(bundle, tuning)
            rows = [score_row(train, predict(bundle, train), 'train'), score_row(tuning, tuning_prediction, 'tuning')]
            part = prediction_part(tuning, tuning_prediction, h, 'tuning')
            part['fold'] = i + 1
            part['fit_labels_before'] = fold['val'][0].normalize()
            part['selection_labels_through'] = folds[-1]['tuning'][-1] + h * STEP
            part['model_info_through'] = part['selection_labels_through']
            part['is_unbiased_oof'] = False
            tuning_parts.append(part)
            add_report(rows, 'search', h, trial.number, i + 1)
            metrics.append(rows[-1]['метрика конкурса'])
            if bundle['trees'] is not None:
                tree_counts.append(bundle['trees'])
            del bundle, train, early, tuning
            gc.collect()
        trial.set_user_attr('tree_counts', tree_counts)
        trial.set_user_attr('fold_scores', metrics)
        value = float(np.mean(metrics))
        if not np.isfinite(value):
            raise ValueError('Неконечная метрика подбора')
        completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
        if not completed or value < min(item.value for item in completed):
            pd.concat(tuning_parts, ignore_index=True).to_parquet(POSTPROCESS_DIR / f'tuning_predictions_h{h}.parquet', index=False)
        return value
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED),
                                storage=f'sqlite:///{(POSTPROCESS_DIR / "search.sqlite3").resolve().as_posix()}',
                                study_name=f'h{h}', load_if_exists=False)
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True)
    study.trials_dataframe().to_csv(DOCS_DIR / f'trials_h{h}.csv', index=False)
    counts = study.best_trial.user_attrs['tree_counts']
    selected = dict(algorithm=ALGORITHM, residual=RESIDUAL, horizon=h, params=study.best_params, trees=int(np.median(counts)) if counts else None,
                    selection_end_exclusive=str(selection_end), selection_labels_through=str(folds[-1]['tuning'][-1] + h * STEP),
                    trial=study.best_trial.number, tuning_score=study.best_value,
                    fold_scores=study.best_trial.user_attrs['fold_scores'])
    (PARAMS_DIR / f'best_h{h}.json').write_text(json.dumps(selected, ensure_ascii=False, indent=2))
    return selected

## Подбор параметров


In [ ]:
if ALGORITHM == 'prophet':
    full_rows = pq.ParquetFile(OUT / 'history_h1.parquet').metadata.num_rows
    width = len(pq.read_schema(OUT / 'history_h1.parquet').names) - 6 + len(route_bit_columns())
    estimated_gb = full_rows * width * 8 * 6 / 1024 ** 3
    available_gb = psutil.virtual_memory().available / 1024 ** 3
    print(f'Prophet full refit: оценка {estimated_gb:.1f} ГБ, доступно {available_gb:.1f} ГБ')
    if estimated_gb > available_gb * 0.8:
        raise MemoryError('Недостаточно RAM для полного Prophet. Нужна среда с большей памятью; автоматического сокращения данных нет.')

selected = {}
for h in range(1, 9):
    selected[h] = select_horizon(h)
    gc.collect()
(PARAMS_DIR / 'selection_frozen.json').write_text(json.dumps(selected, ensure_ascii=False, indent=2))

## Оценка на test


In [ ]:
evaluation_lags = fit_lags(holdout_start.normalize())
evaluation_lags.to_csv(DOCS_DIR / 'status6_lags_evaluation.csv', index=False)
evaluation_rows = []
for h in range(1, 9):
    history = pd.read_parquet(OUT / f'history_h{h}.parquet')
    history = history.loc[np.isfinite(history.profile28)].copy()
    columns = model_columns(history)
    train = add_correlation_features(history.loc[history.timestamp < holdout_start.normalize()].copy(), h, evaluation_lags)
    local_test = add_correlation_features(history.loc[history.cutoff.isin(folds[-1]['test'])].copy(), h, evaluation_lags)
    if len(train) == 0 or len(local_test) == 0 or train.timestamp.max() >= local_test.cutoff.min().normalize():
        raise ValueError('Нарушена граница независимого test')
    bundle = fit_model(train, columns, selected[h]['params'], trees=selected[h]['trees'])
    rows = []
    for split, frame in [('train', train), ('test', local_test)]:
        pred = predict(bundle, frame)
        rows.append(score_row(frame, pred, split))
        part = prediction_part(frame, pred, h, split)
        part['model_info_through'] = holdout_start.normalize() - STEP
        part['is_unbiased_oof'] = split == 'test'
        part.to_parquet(POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet', index=False)
    add_report(rows, 'independent_evaluation', h)
    evaluation_rows.extend([dict(horizon=h, **row) for row in rows])
    save_bundle(bundle, MODEL_DIR / 'evaluation' / f'h{h}')
    del bundle, history, train, local_test, part, frame
    gc.collect()
pd.DataFrame(evaluation_rows).to_csv(DOCS_DIR / 'independent_metrics_by_horizon.csv', index=False)
aggregate_rows = []
for split in ('train', 'test'):
    part = pd.concat([pd.read_parquet(POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet') for h in range(1, 9)], ignore_index=True)
    aggregate_rows.append(score_row(part, part.prediction, split))
    aggregate_rows.append(score_row(part, part.profile28, split + ' profile28'))
    if split == 'test':
        part.to_parquet(POSTPROCESS_DIR / 'test_predictions.parquet', index=False)
    del part
independent_metrics = pd.DataFrame(aggregate_rows)
display(independent_metrics)
independent_metrics.to_csv(DOCS_DIR / 'independent_metrics.csv', index=False)
profile_control = independent_metrics.copy()
profile_control['model'] = np.where(profile_control['выборка'].str.endswith(' profile28'), 'profile28', ALGORITHM)
profile_control['выборка'] = profile_control['выборка'].str.replace(' profile28', '', regex=False)
profile_control.to_csv(DOCS_DIR / 'profile_control.csv', index=False)
print(manifest['test_note'])
export_results()

## Переобучение на всей истории и submission


In [ ]:
full_lags = fit_lags(end + STEP)
full_lags.to_csv(DOCS_DIR / 'status6_lags_full.csv', index=False)
submission_parts, importance_parts = [], []
for h in range(1, 9):
    history = pd.read_parquet(OUT / f'history_h{h}.parquet')
    history = history.loc[np.isfinite(history.profile28)].copy()
    history = add_correlation_features(history, h, full_lags)
    columns = model_columns(history)
    future = add_correlation_features(pd.read_parquet(OUT / f'submission_h{h}.parquet'), h, full_lags)
    bundle = fit_model(history, columns, selected[h]['params'], trees=selected[h]['trees'])
    save_bundle(bundle, MODEL_DIR / 'full' / f'h{h}')
    rows = [score_row(history, predict(bundle, history), 'вся история после refit')]
    add_report(rows, 'full_refit', h)
    display(pd.DataFrame([row for row in evaluation_rows if row['horizon'] == h]))
    print(manifest['test_note'])
    part = future[['id']].copy()
    part['y_pred'] = predict(bundle, future)
    submission_parts.append(part)
    if ALGORITHM in ('xgb', 'catboost', 'lightgbm', 'random_forest'):
        model = bundle['model']
        names = columns
        if ALGORITHM == 'random_forest':
            names = model.named_steps['preprocess'].get_feature_names_out()
            model = model.named_steps['model']
        importance_values = pd.Series(model.feature_importances_, index=names, name=f'h{h}').sort_values(ascending=False)
        importance_values.to_csv(IMPORTANCE_DIR / f'importance_h{h}.csv')
        importance_parts.append(importance_values)
    if ALGORITHM == 'ridge':
        model = bundle['model']
        coefficients = pd.Series(model.named_steps['model'].coef_, index=model.named_steps['preprocess'].get_feature_names_out(), name=f'h{h}')
        coefficients.to_csv(IMPORTANCE_DIR / f'coefficients_h{h}.csv')
        importance_parts.append(coefficients.abs())
    if ALGORITHM == 'prophet':
        from prophet.utilities import regressor_coefficients
        coefficients = regressor_coefficients(bundle['model'])
        mapping = dict(zip(bundle['names'], bundle['prep'].get_feature_names_out()))
        coefficients['feature'] = coefficients.regressor.map(mapping)
        coefficients.to_csv(IMPORTANCE_DIR / f'coefficients_h{h}.csv', index=False)
        importance_parts.append(pd.Series(coefficients.coef.to_numpy(), index=coefficients.feature, name=f'h{h}').abs())
    if ALGORITHM in ('xgb', 'catboost', 'lightgbm', 'random_forest', 'ridge'):
        del model
    del bundle, history, future
    gc.collect()
predictions = pd.concat(submission_parts, ignore_index=True)
submission = test[['id']].merge(predictions, on='id', how='left', validate='one_to_one')
if len(predictions) != len(test) or not np.isfinite(submission.y_pred).all() or (submission.y_pred < 0).any():
    raise ValueError('Некорректный submission')
submission.to_csv(DOCS_DIR / 'submission.csv', index=False)
importance_table = pd.concat(importance_parts, axis=1).rename_axis('feature')
importance_table['mean_all_8'] = importance_table.fillna(0).sum(axis=1) / 8
importance_table['importance_kind'] = 'absolute_coefficient' if ALGORITHM in ('ridge', 'prophet') else 'native_feature_importance'
importance_table = importance_table.sort_values('mean_all_8', ascending=False)
importance_table.to_csv(DOCS_DIR / 'feature_importance.csv')
export_results()
print(f'Результаты: {RUN_DIR.resolve()}')
print(f'Лучшие параметры: {PARAMS_DIR.resolve()}')
print(f'Для Git: {RUN_DIR / "reports.zip"}; модели и прогнозы: {RUN_DIR / "results.zip"}')